# Day 13 Capstone — Research Paper Q&A Agent

This notebook runs the full capstone pipeline end-to-end using real PDF content only:

1. Environment and runtime metadata
2. PDF extraction and chunk/index validation
3. Node isolation tests
4. Mandatory 10-question test suite
5. Memory continuity test
6. RAGAS baseline evaluation
7. Final summary with real measured values

In [1]:
import os
import platform
import multiprocessing

# Constrain threads to reduce CPU spikes/OOM risk.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("CPU cores:", multiprocessing.cpu_count())

try:
    import psutil  # type: ignore
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"RAM total: {ram_gb:.2f} GB")
except Exception:
    print("RAM total: unavailable (psutil not installed)")

Python: 3.12.3
Platform: Linux-6.19.11-100.fc42.x86_64-x86_64-with-glibc2.41
CPU cores: 8
RAM total: 7.57 GB


In [2]:
import agent

print("Papers loaded:", list(agent.PAPERS.keys()))
print("Documents in KB:", len(agent.DOCUMENTS))
print("Collection count:", agent.collection.count())
print("Active model candidates:", [m for m in getattr(agent, '_GROQ_MODEL_CANDIDATES', []) if m])

/home/kanishka/Desktop/ResearchPaperQAProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5931.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ENV OK | ✅ LLM OK | ✅ Embedder OK
✅ transformer: 39509 chars | First 500: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz K
✅ bert: 64129 chars | First 500: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova Google AI Language {jacobdevlin,mingweichang,kentonl,kristout}@google.com Abstract We introduce a new language representa- tion model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language repre- 

In [3]:
# Step checks for document/chunk constraints.
assert len(agent.DOCUMENTS) == 15
assert len({doc['id'] for doc in agent.DOCUMENTS}) == 15
assert all('v' not in doc['paper'] for doc in agent.DOCUMENTS)
assert all(len(doc['text'].split()) >= 150 for doc in agent.DOCUMENTS)
print("✅ 15 chunks verified")

# Retrieval gate confirmation.
query = "What is the self-attention mechanism?"
query_emb = agent.embedder.encode([query]).tolist()
results = agent.collection.query(query_embeddings=query_emb, n_results=3)
returned_papers = [m['paper'] for m in results['metadatas'][0]]
assert '1706.03762' in returned_papers
print("✅ RETRIEVAL GATE PASSED — Transformer chunk confirmed in top-3 for self-attention query")
print("Returned topics:", [m['topic'] for m in results['metadatas'][0]])

✅ 15 chunks verified
✅ RETRIEVAL GATE PASSED — Transformer chunk confirmed in top-3 for self-attention query
Returned topics: ['ReAct — Results + Conclusion', 'Attention Is All You Need — Methodology / Architecture', 'BERT — Methodology / Architecture']


In [4]:
agent.run_node_isolation_tests()

✅ memory_node isolation test passed
⚠️ Groq model fallback activated: llama-3.1-8b-instant
✅ router_node isolation test passed
✅ retrieval_node isolation test passed
✅ skip_retrieval_node isolation test passed
✅ tool_node isolation test passed
✅ answer_node isolation test passed
✅ eval_node isolation test passed
✅ save_node isolation test passed


In [5]:
test_summary = agent.run_mandatory_tests()
print(f"\n10-question pass count: {test_summary['pass_count']}/{test_summary['total']}")

Q1: What problem does the Transformer paper solve?
  Route       : retrieve
  Faithfulness: 0.90
  Answer      : The Transformer paper proposes a new simple network architecture, the Transformer, based solely on attention mechanisms,
  Result      : PASS
Q2: What datasets did BERT use for pre-training?
  Route       : retrieve
  Faithfulness: 1.00
  Answer      : According to the paper, BERT used the following datasets for pre-training:

1. BooksCorpus (800M words) (Zhu et al., 201
  Result      : PASS
Q3: How does RAG combine retrieval with generation?
  Route       : retrieve
  Faithfulness: 1.00
  Answer      : (f(x) · f(z)) where f(x) and f(z) are the dense embeddings of the input query x and the document z, respectively. The re
  Result      : PASS
Q4: What is the ReAct prompting strategy?
  Route       : retrieve
  Faithfulness: 1.00
  Answer      : [Paper: 2210.03629 | Topic: ReAct — Methodology / Architecture] reasoning traces and task-speciﬁc actions in an interlea
  Result   

In [6]:
memory_passed = agent.run_memory_test()
print("Memory test status:", "PASS" if memory_passed else "FAIL")

✅ MEMORY TEST PASSED
Memory test status: PASS


In [7]:
ragas_summary = agent.run_ragas_baseline()
means = ragas_summary['means']
print("RAGAS mode:", means.get('method', 'ragas'))
print("Mean Faithfulness:", f"{means['faithfulness']:.3f}")
print("Mean Answer Relevancy:", f"{means['answer_relevancy']:.3f}")
print("Mean Context Precision:", f"{means['context_precision']:.3f}")

RAGAS evaluation failed, using fallback LLM scoring: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
manual_faithfulness manual_answer_relevancy manual_context_precision
1.000 1.000 1.000
0.933 0.867 0.800
0.333 0.667 0.333
0.933 0.967 0.800
0.667 0.933 0.933

Mean Faithfulness   : 0.773
Mean Ans Relevancy  : 0.887
Mean Context Prec.  : 0.773
RAGAS mode: fallback_llm
Mean Faithfulness: 0.773
Mean Answer Relevancy: 0.887
Mean Context Precision: 0.773


In [8]:
from IPython.display import Markdown, display

summary_md = f"""
## Capstone Summary

| Field              | Detail                            |
|--------------------|-----------------------------------|
| Domain             | Research Paper Q&A                |
| User               | PhD students & researchers        |
| Papers in KB       | 5 real ArXiv PDFs, 15 chunks      |
| Tool 1             | Datetime                          |
| Tool 2             | ArXiv live search (ElementTree)   |
| RAGAS Faithfulness | {means['faithfulness']:.3f} |
| RAGAS Relevancy    | {means['answer_relevancy']:.3f} |
| RAGAS Precision    | {means['context_precision']:.3f} |
| 10-Q Tests         | {test_summary['pass_count']}/{test_summary['total']} PASS |
| Memory Test        | {'PASS' if memory_passed else 'FAIL'} |

**What the agent does:**
This agent routes each user query to retrieval, tools, or memory-only flow, then answers using only retrieved paper excerpts or tool output. It applies faithfulness evaluation and retry logic to stay grounded. It supports session continuity for user name and paper-specific filtering in one thread.

**One specific improvement with more time:**
Replace the manual 3-chunk split per paper with a 200-token sliding window chunker and 50-token overlap using RecursiveCharacterTextSplitter so retrieval is more granular and less likely to miss mid-section findings.
"""

display(Markdown(summary_md))


## Capstone Summary

| Field              | Detail                            |
|--------------------|-----------------------------------|
| Domain             | Research Paper Q&A                |
| User               | PhD students & researchers        |
| Papers in KB       | 5 real ArXiv PDFs, 15 chunks      |
| Tool 1             | Datetime                          |
| Tool 2             | ArXiv live search (ElementTree)   |
| RAGAS Faithfulness | 0.773 |
| RAGAS Relevancy    | 0.887 |
| RAGAS Precision    | 0.773 |
| 10-Q Tests         | 10/10 PASS |
| Memory Test        | PASS |

**What the agent does:**
This agent routes each user query to retrieval, tools, or memory-only flow, then answers using only retrieved paper excerpts or tool output. It applies faithfulness evaluation and retry logic to stay grounded. It supports session continuity for user name and paper-specific filtering in one thread.

**One specific improvement with more time:**
Replace the manual 3-chunk split per paper with a 200-token sliding window chunker and 50-token overlap using RecursiveCharacterTextSplitter so retrieval is more granular and less likely to miss mid-section findings.


## Acceptance Checklist

- Environment loaded from .env and model fallback active when needed
- 5 PDFs extracted and previewed
- 15 chunks indexed in ChromaDB
- Retrieval gate passed for self-attention query
- 8 node isolation tests executed
- Graph compile confirmed
- 10-question suite executed with route + faithfulness + PASS/FAIL
- Red-team prompts tested
- Memory continuity test executed
- RAGAS baseline executed (or fallback scorer if RAGAS unavailable)
- Streamlit app file prepared for launch